Changes from previous crawler system:
- crawler stayed the same
- chunking changed so that it has overlap among chunks and so that each chunk fits the context window of the small embedding model (256 tokens ~ 150-200 words ~ 800 characters)
- embedding model changed (previous one was not good with languages other than english)

In [1]:
import requests   # e "browserul" codului; folosim pentru a trimite cereri HTTP si a primi raspunsul
from bs4 import BeautifulSoup  # instrument de parsing; BS transforma html brut intr un format lizibil pt Python
from urllib.parse import urljoin, urlparse
#urljoin: lipeste fragmentele de link. De ex. daca esti pe google.com si gasesti un link catre /images,
#         il transforma automat in linkul complet: https://google.com/images
#urlparse: exact opusul; descompune un URL in bucatele (protocol, domeniu, cale), ceea ce ajuta la
#          a identifica exact de unde vine un link
import io
import PyPDF2   # pentru a citi si din pdf-uri, nu doar text html


visited = set()
MAX_DEPTH = 3

# Verifica daca linkul "link" gasit pe o alta pagina apartine aceluiasi site de baza, cel de Admitere UBB,
# sau daca este un link extern care te scoate de pe site.
# Literalmente: descompune ambele linkuri si le compara componenta de domeniu (admitere.ubbcluj.ro)
def is_internal(link, base):
    return urlparse(link).netloc == urlparse(base).netloc


def extract_text(html):
    soup = BeautifulSoup(html, "html.parser") #transforma textul brut intr-un obiect structurat (un "copac" de elemente)

    # Nu avem nevoie de textul din interiorul tagurilor urmatoare, deci le stergem cu totul
    for tag in soup(["script", "style", "nav", "footer"]): 
        tag.decompose()

    # get_text ne da "Hello!" in loc de "<p>Hello!</p>"
    return soup.get_text(separator=" ", strip=True)

def extract_text_from_pdf(content):
    pdf_file = io.BytesIO(content)  # transforma bytes in format "fisier in memorie"
    reader = PyPDF2.PdfReader(pdf_file)

    text = ""
    for page in reader.pages:
        text += page.extract_text() + " "

    return text 


def extract_links(html, base_url):
    soup = BeautifulSoup(html, "html.parser")
    links = []

    # Gaseste sectiunea principala a paginii pt a evita linkuri irelevante
    main_content = soup.find("div", class_="entry clearfix")

    if not main_content:
        return []

    for a in main_content.find_all("a", href=True): # cauta toate linkurile (eticheta <a>)
        link = urljoin(base_url, a["href"]) # pentru cazul ca e link relativ; daca in pagina scrie doar <a href="contact.html">, il transforma in https://admitere.ubbcluj.ro/contact.html
        if is_internal(link, base_url):
            links.append(link)

    return links


def crawl(url, depth=0):
    # sa nu accesam prea multe sublinkuri & sa nu le repetam pe aceleasi
    if depth > MAX_DEPTH or url in visited:
        return []

    visited.add(url)

    try:
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/119.0.0.0"} # ca sa nu ne blocheze serverul crezand ca suntem un bot
        response = requests.get(url, headers=headers, timeout=10, stream=True)  # stream=True: descarca intai informatiile despre fisier si decide dupa daca l si descarca pt a i extrage textul

        # --VERIFICARE: Vrem ori HTML, ori pdf
        content_type = response.headers.get('Content-Type', '').lower()

        # Cazul 1: Este pagina HTML
        if 'text/html' in content_type:
            print('Vizitez url: ',url)
            html = response.text        # textul insusi, cu taguri cu tot
            text = extract_text(html)   # textul fara taguri si filtrat
            results = [text]
            for link in extract_links(html, url):
                results += crawl(link, depth + 1)

            return results
        
        # # Cazul 2: Este pdf
        elif 'application/pdf' in content_type:
            print(f"Extrag din pdf: ", url)
            text = extract_text_from_pdf(response.content)  # folosim .content (bytes), nu text
            return [text]

        # Cazul 3: Orice altceva, ignoram
        else:
            return []
        # --
        
    except Exception as e:
        print(f"Eroare la {url}: {e}")
        return []

In [2]:
docs = crawl("https://www.cs.ubbcluj.ro/admitere/nivel-licenta/admiterea-la-facultatea-de-matematica-si-informatica-nivel-licenta/")

Vizitez url:  https://www.cs.ubbcluj.ro/admitere/nivel-licenta/admiterea-la-facultatea-de-matematica-si-informatica-nivel-licenta/
Extrag din pdf:  https://www.cs.ubbcluj.ro/wp-content/uploads/Regulament-admitere-2026-UBB.pdf
Vizitez url:  https://www.cs.ubbcluj.ro/admitere/nivel-licenta/tematica-admitere-facultate/
Extrag din pdf:  https://www.cs.ubbcluj.ro/wp-content/uploads/anexa3a-Regulament-admitere-licenta-2026.pdf
Extrag din pdf:  https://www.cs.ubbcluj.ro/wp-content/uploads/anexa3b-Regulament-admitere-licenta-2026.pdf
Extrag din pdf:  https://www.cs.ubbcluj.ro/wp-content/uploads/anexa2-Regulament-admitere-licenta-2026.pdf
Extrag din pdf:  https://www.cs.ubbcluj.ro/wp-content/uploads/anexa1-Regulament-admitere-licenta-2026.pdf
Vizitez url:  https://www.cs.ubbcluj.ro/admitere/nivel-licenta/taxe-admitere-facultate-nivel-licenta/
Vizitez url:  https://www.cs.ubbcluj.ro/admitere/nivel-licenta/calendarul-concursului-de-admitere-nivel-licenta/


In [4]:
# CHUNKING 2.0
import numpy as np

def chunk_text_with_overlap(text, chunk_size=800, chunk_overlap=150):
    """
    Împarte textul în bucăți de dimensiune fixă (caractere) cu suprapunere.
    chunk_size: lungimea aproximativă a unui fragment.
    chunk_overlap: câți de caractere se repetă din fragmentul anterior.
    """
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    
    while start < len(text):
        # Determinăm sfârșitul ferestrei
        end = start + chunk_size
        
        # Dacă nu suntem la finalul textului, încercăm să nu tăiem un cuvânt la mijloc
        if end < len(text):
            # Căutăm ultimul spațiu în fragment pentru a tăia curat
            last_space = text.rfind(' ', start, end)
            if last_space != -1:
                end = last_space
        
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
            
        # Avansăm cursorul scăzând overlap-ul
        start = end - chunk_overlap
        
        # Siguranță: dacă overlap-ul e prea mare și start nu avansează, forțăm avansul
        if start >= end:
            start = end + 1
            
    return chunks

In [6]:
import re

def clean_text(text):
    # Elimină spațiile multiple și newline-urile redundante
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

chunks = []
for doc in docs:
    # 1. Curățăm textul (foarte important pentru PDF-uri!)
    cleaned_doc = clean_text(doc)
    
    # 2. Aplicăm chunking-ul cu overlap
    # Folosim chunk_size=800 pentru a nu depăși fereastra modelului de embedding
    doc_chunks = chunk_text_with_overlap(cleaned_doc, chunk_size=800, chunk_overlap=150)
    
    chunks.extend(doc_chunks)

print(f"Total chunks în baza de date: {len(chunks)}")

Total chunks în baza de date: 230


In [10]:
# EMBEDDING

from sentence_transformers import SentenceTransformer
import faiss  # baza de date specializata in viteza. Daca ai 1 milion de fragmente de text, FAISS (Facebook AI Similarity Search) gaseste instant care dintre ele seamana cu intrebarea ta
import numpy as np  # FAISS lucreaza doar cu structuri de date de tip numpy

#if model not in globals():   # daca nu am definit pana acum model
model = SentenceTransformer("intfloat/multilingual-e5-small")

# Adaugă prefixul obligatoriu pentru modelul E5 la indexare
chunks_with_prefix = [f"passage: {c}" for c in chunks]
embeddings = model.encode(chunks)

# Creeaza un index (memorie rapida) care masoara distanta "L2" (distanta euclidiana, ca o linie dreapta pe harta) intre vectori
index = faiss.IndexFlatL2(len(embeddings[0]))  # len(embeddings[0]) -> ii spune indexului cat de lung este un singur vector (pentru acest model in specific lungimea fiind de 384 de numere)
index.add(np.array(embeddings))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19209.37it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# SAVE
np.save("chunks.npy", chunks)
faiss.write_index(index, "index.faiss")